In [ ]:
library(Seurat)

In [2]:
library(Matrix)

In [ ]:
library(dplyr)

In [4]:
data <-readRDS('GCA_obj_subset_excludecells3.Rdata')

In [11]:
data@meta.data  %>% filter(!Maincelltype %in% c('Stroma'))  %>% rownames()  %>% data[,.] -> data_filter

In [12]:
data_filter@meta.data %>% mutate(Maincelltype = case_when(celltype=='Epi_normal' ~ 'Epi_normal',
                                                        celltype=='Epi_tumor' ~ 'Epi_tumor',
                                                         .default = Maincelltype )) -> meta.data

In [13]:
head(meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,sample,percent.mt,RNA_snn_res.1.2,seurat_clusters,Maincelltype,subcelltype,celltype,temp
,<fct>,<dbl>,<dbl>,<chr>,<dbl>,<fct>,<fct>,<chr>,<fct>,<chr>,<dbl>
AAACCCAAGACTCTTG-1-SC005,SeuratProject,1864,1114,SC005,4.560086,12,12,Lymphocyte,Tcells,CD4+ T,5.0
AAACCCAAGCAACTTC-1-SC005,SeuratProject,2869,825,SC005,1.707912,0,0,Lymphocyte,Plasma,Plasma,0.1
AAACCCAAGCGTTGTT-1-SC005,SeuratProject,1095,732,SC005,2.557078,40,40,Lymphocyte,Tcells,Regulatory T,0.0
AAACCCAAGTACTGGG-1-SC005,SeuratProject,1236,808,SC005,3.236246,57,57,Lymphocyte,Tcells,NKcell,6.0
AAACCCACAAGCTGCC-1-SC005,SeuratProject,667,508,SC005,2.548726,50,50,Lymphocyte,Tcells,Regulatory T,3.0
AAACCCAGTAAGGCCA-1-SC005,SeuratProject,1140,669,SC005,1.491228,18,18,Phagocyte,MyeloidCells,Neutrophils,0.1


In [15]:
data_filter@meta.data$Maincelltype <- meta.data$Maincelltype

In [16]:
genepos <- read.table('hg38_gencode_v27.txt')

In [5]:
lownames <- readRDS(file = "lownames.rds")
highnames <- readRDS(file = "highnames.rds")

In [9]:
data@meta.data  %>% filter(sample %in% highnames) %>% filter(celltype == 'Epi_tumor')  %>% rownames() -> highcellbarcodes

In [10]:
data@meta.data  %>% filter(sample %in% lownames) %>% filter(celltype == 'Epi_tumor')  %>% rownames() -> lowcellbarcodes

In [12]:
saveRDS(highcellbarcodes,file = "highcellbarcodes.Rds")
saveRDS(lowcellbarcodes,file = "lowcellbarcodes.Rds")

In [18]:
#data_filter@meta.data  %>% filter(sample %in% lownames)  %>% rownames() %>% data2[,.] -> data_low
#data_filter@meta.data  %>% filter(sample %in% highnames)  %>% rownames() %>% data2[,.] -> data_high

In [21]:
data_filter@meta.data  %>% filter(!celltype %in% c('Epi_tumor'))  %>% 
            rownames() %>% data_filter[,.] -> data_normal

In [22]:
data_filter@meta.data  %>% filter(!celltype %in% c('Epi_normal'))  %>% 
            rownames() %>% data_filter[,.] -> data_tumor

In [23]:
rownames(genepos) <- genepos$V1

In [24]:
gene_names <- rownames(data@assays$RNA$counts)

In [25]:
gene_names  %>% intersect(genepos$V1)  %>% genepos[.,] -> genepositions

In [26]:
std_chrom <- paste0('chr',1:22)

In [27]:
genepositions   %>% filter(V2 %in% std_chrom) -> genepositions2

In [28]:
genepositions2$V1  %>% length()

[1] 31849

In [74]:
#countmatrix_low <- data_low@assays$RNA$counts[genepositions2$V1,]

In [29]:
countmatrix_normal <- data_normal@assays$RNA$counts[genepositions2$V1,]

In [30]:
countmatrix_tumor <- data_tumor@assays$RNA$counts[genepositions2$V1,]

In [75]:
#countmatrix_high <- data_high@assays$RNA$counts[genepositions2$V1,]

In [32]:
saveRDS(round(countmatrix_normal, digits=3), "sc.10x.counts.normal.matrix.rds")
saveRDS(round(countmatrix_tumor, digits=3), "sc.10x.counts.tumor.matrix.rds")

In [33]:
dim(countmatrix_normal)

[1] 31849 32318

In [34]:
dim(countmatrix_tumor)

[1] 31849 35874

In [35]:
write.table(x = genepositions2,file = "Gene.pos.txt",quote = F,sep = "\t",row.names = F,col.names = F)

In [36]:
data_normal@meta.data  %>% dplyr::select(Maincelltype) -> annotation_normal
data_tumor@meta.data  %>% dplyr::select(Maincelltype) -> annotation_tumor

In [80]:
dim(annotation_low)

[1] 13138     1

In [81]:
dim(countmatrix_low)

[1] 31849 13138

In [86]:
annotation_low$Maincelltype  %>% unique()

[1] "Lymphocyte" "Epi_tumor"  "Epi_normal"

In [38]:
write.table(x = annotation_normal,file = "annotation_normal.txt",quote = F,sep = "\t",row.names = T,col.names = F)

In [37]:
write.table(x = annotation_tumor,file = "annotation_tumor.txt",quote = F,sep = "\t",row.names = T,col.names = F)